
# COLMAP 내부 동작

> 기준 COLMAP commit: `c004ddf51ee90bb70cbc2bb94a314c7f932febbe` (references/colmap)

convert.py가 호출하는 COLMAP 명령어들이 내부적으로 어떻게 동작하는지 설명합니다.

## 전체 흐름

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        COLMAP Pipeline (convert.py)                         │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  [1] feature_extractor ──────────────────────────────────────────────────── │
│      ├─ 1.1 Image reading + EXIF parsing                                    │
│      ├─ 1.2 K (Intrinsic) 초기 추정                                         │
│      ├─ 1.3 Camera model 할당 (OPENCV)                                      │
│      ├─ 1.4 SIFT keypoint 검출 (DoG)                                        │
│      └─ 1.5 SIFT descriptor 계산 (128-dim)                                  │
│      └─→ database.db에 cameras, keypoints, descriptors 저장                 │
│                                                                             │
│  [2] exhaustive_matcher ─────────────────────────────────────────────────── │
│      ├─ 2.1 Pair generation (모든 이미지 쌍)                                │
│      ├─ 2.2 Descriptor matching (KNN + Ratio test)                          │
│      └─ 2.3 Geometric Verification                                          │
│          ├─ E/F/H 추정 (5-point + RANSAC)                                   │
│          └─ Inlier filtering                                                │
│      └─→ database.db에 matches, two_view_geometries 저장                    │
│                                                                             │
│  [3] mapper (Incremental SfM) ───────────────────────────────────────────── │
│      ├─ 3.1 Database cache loading                                          │
│      ├─ 3.2 Initial Pair Selection                                          │
│      ├─ 3.3 Two-View Geometry (E → R, t)                                    │
│      ├─ 3.4 Initial Triangulation                                           │
│      ├─ 3.5 Initial Global Bundle Adjustment                                │
│      └─ 3.6 [반복] Incremental Registration                                 │
│          ├─ Find next image                                                 │
│          ├─ PnP (P3P + RANSAC)                                              │
│          ├─ Triangulation                                                   │
│          ├─ Local Bundle Adjustment                                         │
│          └─ Filter outliers                                                 │
│      └─ 3.7 Final Global Bundle Adjustment                                  │
│      └─→ sparse/0/에 cameras.bin, images.bin, points3D.bin 저장             │
│                                                                             │
│  [4] image_undistorter ──────────────────────────────────────────────────── │
│      ├─ 4.1 Distortion model 적용                                           │
│      └─ 4.2 OPENCV → PINHOLE 변환                                           │
│      └─→ undistorted 이미지 + 새 카메라 파라미터 저장                       │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```



---

## Step 1: Feature Extraction

### 1.1 Image Reading + EXIF Parsing

In [ ]:
// src/colmap/controllers/image_reader.cc
ImageReader::Status ImageReader::Next(...) {
  // EXIF에서 focal length 추출 시도
  const std::optional<double> maybe_focal_length = bitmap->ExifFocalLength();
}



### 1.2 K (Intrinsic) 초기 추정

Feature extraction 단계에서 **카메라 intrinsic이 가장 먼저 추정**됩니다.

In [ ]:
// src/colmap/controllers/image_reader.cc
const double focal_length = maybe_focal_length.value_or(
    options_.default_focal_length_factor *
    std::max(bitmap->Width(), bitmap->Height()));

prev_camera_ = Camera::CreateFromModelId(prev_camera_.camera_id,
                                         prev_camera_.model_id,
                                         focal_length,
                                         bitmap->Width(),
                                         bitmap->Height());



| 상황 | 추정 방법 |
|------|-----------|
| **EXIF 있음** | $f = \frac{\text{focal\_mm}}{\text{sensor\_mm}} \times \text{width}$ |
| **EXIF 없음** | $f = 1.2 \times \max(W, H)$ (휴리스틱) |

**Principal point**는 항상 이미지 중심으로 초기화: $(c_x, c_y) = (W/2, H/2)$

### 1.3 Camera Model 할당

In [ ]:
// src/colmap/sensor/models.cc
std::vector<double> OpenCVCameraModel::InitializeParams(
    const double focal_length, const size_t width, const size_t height) {
  return {focal_length, focal_length, width / 2.0, height / 2.0, 0, 0, 0, 0};
  //         fx            fy             cx           cy        k1 k2 p1 p2
}



3DGS `convert.py`는 `--ImageReader.camera_model OPENCV`를 사용하여 8파라미터 왜곡 모델을 사용합니다.

### 1.4-1.5 SIFT Feature Detection & Descriptor

In [ ]:
// src/colmap/feature/sift.cc
class SiftCPUFeatureExtractor : public FeatureExtractor {
  bool Extract(const Bitmap& bitmap,
               FeatureKeypoints* keypoints,
               FeatureDescriptors* descriptors) {
    // VLFeat SIFT: Scale-space pyramid → DoG → Keypoint detection
    vl_sift_set_peak_thresh(sift_.get(), options_.sift->peak_threshold);
    vl_sift_set_edge_thresh(sift_.get(), options_.sift->edge_threshold);
    
    while (true) {
      vl_sift_detect(sift_.get());
      const VlSiftKeypoint* vl_keypoints = vl_sift_get_keypoints(sift_.get());
      // 128차원 descriptor 계산
    }
  }
};



| convert.py 옵션 | COLMAP 내부 |
|-----------------|-------------|
| `--ImageReader.single_camera 1` | 모든 이미지에 동일 camera_id |
| `--ImageReader.camera_model OPENCV` | 8파라미터 왜곡 모델 |
| `--SiftExtraction.use_gpu 1` | SiftGPU 사용 |

---

## Step 2: Feature Matching

### 2.1 Pair Generation

Exhaustive matcher는 모든 이미지 쌍을 비교합니다: $N$개 이미지 → $\frac{N(N-1)}{2}$개 쌍

### 2.2 Descriptor Matching

In [ ]:
// src/colmap/feature/sift.cc
bool SiftMatchingOptions::Check() const {
  CHECK_OPTION_GT(max_ratio, 0.0);      // Lowe's ratio test threshold
  CHECK_OPTION_GT(max_distance, 0.0);   // Maximum descriptor distance
  return true;
}



**Lowe's Ratio Test:**
$$\frac{d_{\text{best}}}{d_{\text{second best}}} < 0.8$$

### 2.3 Geometric Verification

Feature matching 후, **기하학적 검증**을 통해 outlier를 제거합니다.

In [ ]:
// src/colmap/estimators/two_view_geometry.cc
void TwoViewGeometry::Estimate(const Camera& camera1,
                               const std::vector<Eigen::Vector2d>& points1,
                               const Camera& camera2,
                               const std::vector<Eigen::Vector2d>& points2,
                               const FeatureMatches& matches,
                               const Options& options) {
  // Essential Matrix 추정 (calibrated cameras)
  // 또는 Fundamental Matrix (uncalibrated)
  // 또는 Homography (planar scene)
}



**5-Point Algorithm + RANSAC:**

In [ ]:
// src/colmap/estimators/solvers/essential_matrix.cc
void EssentialMatrixFivePointEstimator::Estimate(
    const std::vector<X_t>& points1,
    const std::vector<Y_t>& points2,
    std::vector<M_t>* models) {
  // 정규화된 좌표에서 E 추정
  // Epipolar constraint: x2' * E * x1 = 0
}



| convert.py 옵션 | COLMAP 내부 |
|-----------------|-------------|
| `--SiftMatching.use_gpu 1` | GPU 가속 descriptor 비교 |

---

## Step 3: Mapper (Incremental SfM)

### 3.1 Database Cache Loading

In [ ]:
// src/colmap/sfm/incremental_mapper.cc
IncrementalMapper::IncrementalMapper(
    std::shared_ptr<const DatabaseCache> database_cache)
    : database_cache_(std::move(database_cache)) {}



### 3.2 Initial Pair Selection

In [ ]:
// src/colmap/sfm/incremental_mapper_impl.cc
std::vector<image_t> IncrementalMapperImpl::FindFirstInitialImage(...) {
  // 선택 기준:
  // 1. Prior focal length (EXIF) 있는 이미지 우선
  // 2. 많은 correspondence를 가진 이미지
  // 3. 이전에 초기화 시도하지 않은 이미지
}



**좋은 초기 쌍의 조건:**

| 조건 | 기본값 | 이유 |
|------|--------|------|
| 최소 inlier 수 | 100 | 안정적인 추정 |
| 최소 삼각측량 각도 | 16° | 깊이 추정 정확도 |
| 최대 forward motion | 0.95 | degenerate 방지 |

### 3.3 Two-View Geometry (E → R, t)

In [ ]:
// src/colmap/sfm/incremental_mapper.cc
void IncrementalMapper::RegisterInitialImagePair(
    const Options& options,
    const image_t image_id1,
    const image_t image_id2,
    const Rigid3d& cam2_from_cam1) {
  // 첫 번째 카메라: identity
  image1.FramePtr()->SetCamFromWorld(image1.CameraId(), Rigid3d());
  // 두 번째 카메라: 상대 포즈 적용
  image2.FramePtr()->SetCamFromWorld(image2.CameraId(), cam2_from_cam1);
}



**E → R, t 분해 (SVD):**
$$E = U \Sigma V^T \Rightarrow R, t \text{ (4가지 해 중 1개 선택)}$$

### 3.4 Initial Triangulation

In [ ]:
// src/colmap/geometry/triangulation.cc
bool TriangulatePoint(const Eigen::Matrix3x4d& cam1_from_world,
                      const Eigen::Matrix3x4d& cam2_from_world,
                      const Eigen::Vector2d& cam_point1,
                      const Eigen::Vector2d& cam_point2,
                      Eigen::Vector3d* xyz) {
  // DLT (Direct Linear Transform)
  Eigen::Matrix4d A;
  A.row(0) = cam_point1(0) * cam1_from_world.row(2) - cam1_from_world.row(0);
  A.row(1) = cam_point1(1) * cam1_from_world.row(2) - cam1_from_world.row(1);
  A.row(2) = cam_point2(0) * cam2_from_world.row(2) - cam2_from_world.row(0);
  A.row(3) = cam_point2(1) * cam2_from_world.row(2) - cam2_from_world.row(1);
  
  // SVD로 null space 찾기
  const Eigen::JacobiSVD<Eigen::Matrix4d> svd(A, Eigen::ComputeFullV);
  *xyz = svd.matrixV().col(3).hnormalized();
}



### 3.5 Initial Global Bundle Adjustment

모든 카메라 포즈, intrinsics, 3D 점을 동시에 최적화:

$$\min \sum_{i,j} \rho\left(\|\mathbf{x}_{ij} - \pi(K_i, R_i, t_i, \mathbf{X}_j)\|^2\right)$$

### 3.6 [반복] Incremental Registration

#### PnP (Perspective-n-Point)

In [ ]:
// src/colmap/sfm/incremental_mapper.cc
bool IncrementalMapper::RegisterNextImage(const Options& options,
                                          const image_t image_id) {
  // 2D-3D correspondences 수집
  if (obs_manager_->NumVisiblePoints3D(image_id) <
      static_cast<size_t>(options.abs_pose_min_num_inliers)) {
    return false;  // 충분한 대응점이 없음
  }
  
  // P3P + RANSAC으로 카메라 포즈 추정
  // ...
}



**P3P Algorithm:**
- 3개의 3D-2D 대응으로 카메라 포즈 추정
- 최대 4개의 해 → RANSAC으로 best 선택

#### Triangulation

새 이미지 등록 후, 새로운 3D 점 생성:

In [ ]:
// src/colmap/sfm/incremental_triangulator.cc
size_t IncrementalTriangulator::TriangulateImage(const Options& options,
                                                  const image_t image_id) {
  // 기존 track 연장 또는 새 3D 점 생성
}



#### Local Bundle Adjustment

In [ ]:
// src/colmap/sfm/incremental_mapper.cc
void IncrementalMapper::AdjustLocalBundle(const Options& options, ...) {
  // 가장 연결된 이웃 이미지들과 함께 최적화
  // ba_local_num_images = 6 (기본값)
}



### 3.7 Final Global Bundle Adjustment

In [ ]:
// src/colmap/estimators/bundle_adjustment_ceres.cc
// Ceres Solver + Levenberg-Marquardt
// Schur Complement로 효율적인 계산



### Mapper 옵션 정리

In [ ]:
// src/colmap/sfm/incremental_mapper.h
struct Options {
  // Initial pair
  int init_min_num_inliers = 100;
  double init_max_error = 4.0;
  double init_min_tri_angle = 16.0;
  double init_max_forward_motion = 0.95;
  
  // PnP
  double abs_pose_max_error = 12.0;
  int abs_pose_min_num_inliers = 30;
  double abs_pose_min_inlier_ratio = 0.25;
  
  // Bundle Adjustment
  int ba_local_num_images = 6;
  
  // Filtering
  double filter_max_reproj_error = 4.0;
  double filter_min_tri_angle = 1.5;
};



| convert.py 옵션 | COLMAP 내부 |
|-----------------|-------------|
| `--Mapper.ba_global_function_tolerance=0.000001` | BA 수렴 조건 |

---

## Step 4: Image Undistortion

### 4.1-4.2 Distortion 제거 및 PINHOLE 변환

In [ ]:
// src/colmap/exe/image.cc
int RunImageUndistorter(int argc, char** argv) {
  Reconstruction reconstruction;
  reconstruction.Read(input_path);
  
  std::unique_ptr<BaseController> undistorter;
  if (output_type == "COLMAP") {
    undistorter = std::make_unique<COLMAPUndistorter>(
        undistorter_options,
        undistort_camera_options,
        reconstruction,
        *options.image_path,
        output_path);
  }
  undistorter->Run();
}



**변환 과정:**

```
OPENCV:  fx, fy, cx, cy, k1, k2, p1, p2  (8 params)
    ↓ undistort
PINHOLE: fx, fy, cx, cy                  (4 params)
```



3DGS 렌더러는 **PINHOLE 카메라만 지원**하므로 이 변환이 필수입니다.

---

## 출력 파일 구조

### database.db (SQLite)

| 테이블 | 내용 | 저장 단계 |
|--------|------|-----------|
| `cameras` | 카메라 모델, K 파라미터 | feature_extractor |
| `images` | 이미지 이름, camera_id | feature_extractor |
| `keypoints` | SIFT keypoints | feature_extractor |
| `descriptors` | 128차원 SIFT descriptors | feature_extractor |
| `matches` | 이미지 쌍별 raw matches | exhaustive_matcher |
| `two_view_geometries` | E/F matrix, inliers | exhaustive_matcher |

### sparse/0/ (Binary)

| 파일 | 내용 | 저장 단계 |
|------|------|-----------|
| `cameras.bin` | 최종 K (BA 후) | mapper |
| `images.bin` | 카메라 포즈 (qvec, tvec) | mapper |
| `points3D.bin` | Sparse 3D point cloud | mapper |

---

## 참고: 소스 코드 위치

| 기능 | 파일 경로 |
|------|-----------|
| CLI 진입점 | `src/colmap/exe/feature.cc`, `sfm.cc`, `image.cc` |
| Image reader + K 추정 | `src/colmap/controllers/image_reader.cc` |
| SIFT 추출 | `src/colmap/feature/sift.cc` |
| Feature 매칭 | `src/colmap/feature/matcher.cc` |
| Two-view geometry | `src/colmap/estimators/two_view_geometry.cc` |
| Essential matrix (5-point) | `src/colmap/estimators/solvers/essential_matrix.cc` |
| Incremental SfM | `src/colmap/sfm/incremental_mapper.cc` |
| Triangulation | `src/colmap/geometry/triangulation.cc` |
| PnP (P3P) | `src/colmap/estimators/solvers/absolute_pose.cc` |
| Bundle Adjustment | `src/colmap/estimators/bundle_adjustment_ceres.cc` |
| Undistortion | `src/colmap/controllers/undistorters.cc` |
| Camera models | `src/colmap/sensor/models.cc` |